In [32]:
# %pip install llama-index-agent-openai
# %pip install llama-index-llms-openai


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
# !pip install llama-index


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


# Benchmarking OpenAI Retrieval API (through Assistant Agent)

### This guide benchmarks the Retrieval Tool from the OpenAI Assistant API, by using our OpenAIAssistantAgent. We run over the Llama 2 paper, and compare generation quality against a naive RAG pipeline.

In [7]:
import json
from typing import Sequence, List
from typing import Dict
import pandas as pd

from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import AgentRunner
from llama_index.agent.openai import OpenAIAgentWorker, OpenAIAgent
from llama_index.core.agent import AgentRunner, ReActAgentWorker, ReActAgent
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_tool, trace_llm, trace_agent
from ragaai_catalyst.tracers import Tracer
import json
from typing import Sequence, List

from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import AgentRunner
from llama_index.agent.openai import OpenAIAgentWorker, OpenAIAgent
from llama_index.core.agent import AgentRunner, ReActAgentWorker, ReActAgent
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_tool, trace_llm, trace_agent
from ragaai_catalyst.tracers import Tracer

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "RAGAAI_CATALYST_ACCESS_KEY",
    "RAGAAI_CATALYST_SECRET_KEY",
    "RAGAAI_CATALYST_BASE_URL",
    "OPENAI_API_KEY"
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

# Initialize RagaAI Catalyst with error checking
try:
    catalyst = RagaAICatalyst(
        access_key=os.getenv("RAGAAI_CATALYST_ACCESS_KEY"),
        secret_key=os.getenv("RAGAAI_CATALYST_SECRET_KEY"),
        base_url=os.getenv("RAGAAI_CATALYST_BASE_URL"),
    )
    print("RagaAI Catalyst initialized successfully")
except Exception as e:
    raise Exception(f"Failed to initialize RagaAI Catalyst: {str(e)}")
#create project
# project = catalyst.create_project(
#     project_name="benchmarking_agent",
#     usecase="Agentic Application"
# )
# Initialize tracer with more specific naming
tracer = Tracer(
    project_name="benchmarking_agent",
    dataset_name="benchmarking_agent",
    tracer_type="Agentic",
)

# Initialize tracing with error handling
try:
    init_tracing(catalyst=catalyst, tracer=tracer)
    print("Tracing initialized successfully")
except Exception as e:
    raise Exception(f"Error initializing tracing: {str(e)}")




class BenchmarkingSystem:
    def __init__(self):
        self.results = []
        self.llm = OpenAI(model="gpt-3.5-turbo")  # Initialize OpenAI LLM
        
    @tracer.trace_llm(name="run_llm_query")
    def run_llm_query(self, prompt: str, model: str) -> Dict:
        """
        Run a query through an LLM and measure performance
        """
        start_time = time.time()
        try:
            # Use llama_index's OpenAI wrapper instead of raw completion
            response = self.llm.complete(prompt)
            end_time = time.time()
            
            result = {
                "model": model,
                "prompt": prompt,
                "response": response.text,  # Access text from CompletionResponse
                "latency": end_time - start_time,
                "status": "success"
            }
            
        except Exception as e:
            end_time = time.time()
            result = {
                "model": model,
                "prompt": prompt,
                "response": str(e),
                "latency": end_time - start_time,
                "status": "error"
            }
            
        return result



    @tracer.trace_tool(name="calculate_metrics")
    def calculate_metrics(self, results: List[Dict]) -> Dict:
        """
        Calculate performance metrics from results
        """
        df = pd.DataFrame(results)
        metrics = {
            "avg_latency": df[df["status"] == "success"]["latency"].mean(),
            "success_rate": (df["status"] == "success").mean() * 100,
            "total_queries": len(df),
            "failed_queries": len(df[df["status"] == "error"])
        }
        return metrics

    @tracer.trace_agent(name="benchmark_agent")
    def run_benchmark(self, prompts: List[str], models: List[str]) -> Dict:
        """
        Run benchmark tests across multiple models and prompts
        """
        for model in models:
            for prompt in prompts:
                result = self.run_llm_query(prompt, model)
                self.results.append(result)
                
        metrics = self.calculate_metrics(self.results)
        return {
            "detailed_results": self.results,
            "summary_metrics": metrics
        }

def main():
    # Test prompts
    test_prompts = [
        "Explain quantum computing in simple terms",
        "What are the main challenges in artificial intelligence?",
        "How does blockchain technology work?"
    ]
    
    # Models to test
    test_models = [
        "gpt-3.5-turbo",
        "gpt-4"
    ]
    
    # Initialize benchmarking system
    benchmark_system = BenchmarkingSystem()
    
    # Run benchmarks
    with tracer:
        try:
            print("Starting benchmark tests...")
            results = benchmark_system.run_benchmark(test_prompts, test_models)
            
            print("\nBenchmark Summary:")
            print("-----------------")
            for metric, value in results["summary_metrics"].items():
                print(f"{metric}: {value}")
                
            print("\nDetailed Results:")
            print("----------------")
            for result in results["detailed_results"]:
                print(f"\nModel: {result['model']}")
                print(f"Latency: {result['latency']:.2f}s")
                print(f"Status: {result['status']}")
                
        except Exception as e:
            print(f"Error during benchmarking: {str(e)}")
            
if __name__ == "__main__":
    main()
    tracer.stop()

Token(s) set successfully
RagaAI Catalyst initialized successfully
Tracing initialized successfully
Starting benchmark tests...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ragaai_catalyst.tracers.agentic_tracing.utils.zip_list_of_unique_files: Zip file created successfully.
INFO:ragaai_catalyst.tracers.agentic_tracing.tracers.base: Traces saved successfully.



Benchmark Summary:
-----------------
avg_latency: 2.2905554374059043
success_rate: 100.0
total_queries: 6
failed_queries: 0

Detailed Results:
----------------

Model: gpt-3.5-turbo
Latency: 1.95s
Status: success

Model: gpt-3.5-turbo
Latency: 2.67s
Status: success

Model: gpt-3.5-turbo
Latency: 2.92s
Status: success

Model: gpt-4
Latency: 1.57s
Status: success

Model: gpt-4
Latency: 2.32s
Status: success

Model: gpt-4
Latency: 2.31s
Status: success
Uploading agentic traces...
Uploading code...
Dataset trace code inserted successfully
